In [37]:
import numpy as np

In [38]:
from preprocessing import preprocess
X_train, y_train, X_val, y_val, X_test, y_test, weights = preprocess(feature_method="flatten", n_pca=50)

Loading MNIST dataset...
Split completed: Train=54000, Val=6000, Test=10000


In [39]:
# activation function (y_hat)
def sigmoid(z):
    return 1.0/(1+np.exp(-z))

In [40]:

def compute_loss(y, y_hat, weights):
    epsilon = 1e-15
    y_hat = np.clip(y_hat, epsilon, 1 - epsilon)
    
    loss = - (
        weights[1] * y * np.log(y_hat) +
        weights[0] * (1 - y) * np.log(1 - y_hat)
    )
    
    return np.mean(loss)

In [41]:
def compute_gradients(X, y, y_hat, weights):
    m = len(y)
    # apply weights per sample
    sample_weights = np.where(y == 1, weights[1], weights[0])
    error = (y_hat - y) * sample_weights
    dw = (1/m) * (X.T @ error)
    db = (1/m) * np.sum(error)
    
    return dw, db

In [42]:
def compute_gradient_descent(w, b, dw, db, learning_rate):
    w = w - learning_rate * dw
    b = b - learning_rate * db
    return w, b

In [43]:
def fit(X, y, weights, iterations=1000, lr=0.01):
    
    n_features = X.shape[1]
    w = np.zeros(n_features)
    b = 0
    
    for i in range(iterations):
        z = X @ w + b
        y_hat = sigmoid(z)
        
        # Loss
        loss = compute_loss(y, y_hat, weights)
        
        # compute Gradients ( dL/dw , dL/db)
        dw, db = compute_gradients(X, y, y_hat, weights)
        
        # Update parameters
        w -= lr * dw
        b -= lr * db
        
        if i % 100 == 0:
            print(f"Iteration {i}, Loss: {loss:.4f}")
    
    return w, b

In [44]:
def predict(X,W,b):
    z = X @ W + b
    y_hat=sigmoid(z)
    return (y_hat >= 0.5).astype(int)


In [45]:
# Train model
w, b = fit(X_train, y_train, weights, iterations=400, lr=0.1)
# Validation performance
y_val_pred = predict(X_val, w, b)
# Test performance
y_test_pred = predict(X_test, w, b)

Iteration 0, Loss: 0.6931
Iteration 100, Loss: 0.1175
Iteration 200, Loss: 0.0955
Iteration 300, Loss: 0.0857


In [46]:
def evaluate(X, y, w, b, dataset_name="Validation"):
    y_pred = predict(X, w, b)
    y_true = y

    # Positive class = 0 (digit zero)
    TP = np.sum((y_pred == 0) & (y_true == 0))
    TN = np.sum((y_pred == 1) & (y_true == 1))
    FP = np.sum((y_pred == 0) & (y_true == 1))
    FN = np.sum((y_pred == 1) & (y_true == 0))

    accuracy  = (TP + TN) / (TP + TN + FP + FN)
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall    = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    f1        = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

    print(f"--- {dataset_name} Results ---")
    print(f"Accuracy  : {accuracy:.4f}")
    print(f"Precision : {precision:.4f}")
    print(f"Recall    : {recall:.4f}")
    print(f"F1-Score  : {f1:.4f}")

    print("Confusion Matrix:")
    print(f"                 Predicted 0   Predicted 1")
    print(f"  Actual 0   :   {TP:<12}  {FN}")
    print(f"  Actual 1   :   {FP:<12}  {TN}")

In [47]:
# Evaluate
evaluate(X_val, y_val, w, b, "Validation")
evaluate(X_test, y_test, w, b, "Test")

--- Validation Results ---
Accuracy  : 0.9752
Precision : 0.8120
Recall    : 0.9710
F1-Score  : 0.8844
Confusion Matrix:
                 Predicted 0   Predicted 1
  Actual 0   :   570           17
  Actual 1   :   132           5281
--- Test Results ---
Accuracy  : 0.9756
Precision : 0.8067
Recall    : 0.9878
F1-Score  : 0.8881
Confusion Matrix:
                 Predicted 0   Predicted 1
  Actual 0   :   968           12
  Actual 1   :   232           8788


In [48]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# ===== Validation Metrics =====
print("=== Validation Metrics ===")
print("Accuracy :", accuracy_score(y_val, y_val_pred))
print("Precision:", precision_score(y_val, y_val_pred, pos_label=0))
print("Recall   :", recall_score(y_val, y_val_pred, pos_label=0))
print("F1 Score :", f1_score(y_val, y_val_pred, pos_label=0))


# ===== Test Metrics =====
print("\n=== Test Metrics ===")
print("Accuracy :", accuracy_score(y_test, y_test_pred))
print("Precision:", precision_score(y_test, y_test_pred, pos_label=0))
print("Recall   :", recall_score(y_test, y_test_pred, pos_label=0))
print("F1 Score :", f1_score(y_test, y_test_pred, pos_label=0))


# ===== Classification Report =====
print("\n=== Classification Report (Test) ===")
print(classification_report(y_test, y_test_pred, target_names=["Class 0 (zero)", "Class 1 (not zero)"]))


# ===== Confusion Matrix =====
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

=== Validation Metrics ===
Accuracy : 0.9751666666666666
Precision: 0.811965811965812
Recall   : 0.9710391822827938
F1 Score : 0.8844065166795966

=== Test Metrics ===
Accuracy : 0.9756
Precision: 0.8066666666666666
Recall   : 0.9877551020408163
F1 Score : 0.8880733944954129

=== Classification Report (Test) ===
                    precision    recall  f1-score   support

    Class 0 (zero)       0.81      0.99      0.89       980
Class 1 (not zero)       1.00      0.97      0.99      9020

          accuracy                           0.98     10000
         macro avg       0.90      0.98      0.94     10000
      weighted avg       0.98      0.98      0.98     10000


Confusion Matrix:
[[ 968   12]
 [ 232 8788]]
